# End-to-End Model Pipeline

This notebook walks through the complete workflow: from raw data exploration
to a trained, registered, and deployed anomaly detection model on the OpenUBA platform.

**What you'll do:**
1. Generate synthetic UBA login data with injected anomalies
2. Explore and visualize the dataset
3. Train an Isolation Forest locally in the notebook
4. Register the trained model with the platform via the SDK
5. Verify the model is installed and ready
6. Create a dataset record on the platform
7. Launch a training job on the platform
8. Launch an inference job and retrieve results
9. Visualize risk scores and publish to a dashboard

**SDK functions used:** `register_model`, `load_model`, `create_dataset`,
`start_training`, `start_inference`, `get_job`, `wait_for_job`, `get_logs`,
`publish_version`, `create_visualization`, `create_dashboard`

---
## 1. Setup & Imports

In [ ]:
import openuba
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

print(f'OpenUBA SDK v{openuba.__version__}')
print(f'numpy {np.__version__}, pandas {pd.__version__}')
print('\nSDK auto-configured from workspace environment variables.')

---
## 2. Generate Synthetic Login Data

We simulate 30 days of SSH login activity for 200 users.
Most users have normal behavior; ~5% are injected anomalies
(brute-force attempts, off-hours access, geographic impossibility).

In [ ]:
np.random.seed(42)
n_users = 200
n_anomalous = 10  # ~5% anomaly rate

records = []
for i in range(n_users):
    user_id = f'user_{i:04d}'
    is_anomalous = i < n_anomalous

    if is_anomalous:
        # anomalous users: high login counts, many unique IPs, high fail rate
        records.append({
            'entity_id': user_id,
            'login_count_24h': np.random.randint(50, 200),
            'unique_ips_24h': np.random.randint(10, 50),
            'failed_login_ratio': np.random.uniform(0.4, 0.9),
            'off_hours_activity': np.random.uniform(0.6, 1.0),
            'bytes_transferred_mb': np.random.uniform(500, 5000),
            'session_duration_min': np.random.uniform(0.5, 3.0),
            'new_device_flag': np.random.choice([0, 1], p=[0.2, 0.8]),
            'label': 'anomalous',
        })
    else:
        # normal users: low login counts, few IPs, low fail rate
        records.append({
            'entity_id': user_id,
            'login_count_24h': np.random.randint(1, 15),
            'unique_ips_24h': np.random.randint(1, 4),
            'failed_login_ratio': np.random.uniform(0.0, 0.1),
            'off_hours_activity': np.random.uniform(0.0, 0.2),
            'bytes_transferred_mb': np.random.uniform(5, 100),
            'session_duration_min': np.random.uniform(5, 60),
            'new_device_flag': np.random.choice([0, 1], p=[0.9, 0.1]),
            'label': 'normal',
        })

df = pd.DataFrame(records)
print(f'Dataset shape: {df.shape}')
print(f'Anomalous: {(df.label == "anomalous").sum()}, Normal: {(df.label == "normal").sum()}')
df.head()

---
## 3. Explore the Data

Quick look at feature distributions to confirm the anomaly injection worked.

In [ ]:
feature_cols = [
    'login_count_24h', 'unique_ips_24h', 'failed_login_ratio',
    'off_hours_activity', 'bytes_transferred_mb', 'session_duration_min',
    'new_device_flag',
]

print('=== Normal Users ===')
print(df[df.label == 'normal'][feature_cols].describe().round(2))
print()
print('=== Anomalous Users ===')
print(df[df.label == 'anomalous'][feature_cols].describe().round(2))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
plot_features = ['login_count_24h', 'unique_ips_24h', 'failed_login_ratio',
                 'off_hours_activity', 'bytes_transferred_mb', 'session_duration_min']

for ax, feat in zip(axes.flat, plot_features):
    normal = df[df.label == 'normal'][feat]
    anomalous = df[df.label == 'anomalous'][feat]
    ax.hist(normal, bins=20, alpha=0.6, label='Normal', color='steelblue')
    ax.hist(anomalous, bins=20, alpha=0.6, label='Anomalous', color='crimson')
    ax.set_title(feat)
    ax.legend(fontsize=8)

fig.suptitle('Feature Distributions: Normal vs Anomalous', fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Train the Model Locally

Train an Isolation Forest on the numeric features. This gives us a fitted model
object that we'll register with the platform.

In [ ]:
X = df[feature_cols].values

clf = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples='auto',
    random_state=42,
)
clf.fit(X)

# score locally to verify the model works
predictions = clf.predict(X)
scores = clf.decision_function(X)

df['prediction'] = predictions   # 1 = normal, -1 = anomaly
df['anomaly_score'] = scores

detected = (df.prediction == -1)
true_anomalous = (df.label == 'anomalous')

tp = (detected & true_anomalous).sum()
fp = (detected & ~true_anomalous).sum()
fn = (~detected & true_anomalous).sum()
tn = (~detected & ~true_anomalous).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'Local evaluation on training data:')
print(f'  True Positives:  {tp}')
print(f'  False Positives: {fp}')
print(f'  False Negatives: {fn}')
print(f'  True Negatives:  {tn}')
print(f'  Precision: {precision:.3f}')
print(f'  Recall:    {recall:.3f}')
print(f'  F1 Score:  {f1:.3f}')
print(f'\nModel is fitted with {clf.n_estimators} trees.')

---
## 5. Register the Model with the Platform

This is the key step. `openuba.register_model()` does the following:
- Detects the framework (sklearn) automatically from the model object
- Serializes the fitted model (pickle) and sends it as base64
- The backend writes `MODEL.py`, `model.pkl`, and `model.yaml` to the model library
- Creates DB records (Model, ModelVersion, ModelComponent)
- Sets status to `installed` so the model is immediately available

After registration, the model appears in **Models > Installed** in the UI
and can be trained/inferred by the model runner.

In [ ]:
handle = openuba.register_model(
    name='ssh-login-anomaly-detector',
    model=clf,
    description='Isolation Forest trained on SSH login behavior features. '
                'Detects brute-force attempts, credential stuffing, and '
                'unusual access patterns.',
)

model_id = handle.get('model_id')
print(f'Model registered successfully!')
print(f'  model_id: {model_id}')
print(f'  name:     {handle.get("name")}')
print(f'  version:  {handle.get("version")}')
print(f'  status:   {handle.get("status")}')
print(f'  slug:     {handle.get("slug")}')

---
## 6. Verify the Model is Installed

Use `load_model()` to confirm the platform can resolve the model by name
and that its status is `installed`.

In [ ]:
resolved = openuba.load_model('ssh-login-anomaly-detector')

print('Model resolved from platform:')
for key, value in resolved.items():
    print(f'  {key}: {value}')

assert resolved.get('status') == 'installed', \
    f'Expected status=installed, got {resolved.get("status")}'
print('\nStatus is "installed" — model is ready for platform jobs.')

---
## 7. Create a Dataset on the Platform

Register the dataset so training and inference jobs can reference it.
In production, you'd upload a CSV/Parquet file; here we create the record.

In [ ]:
# save the data locally (the model runner will mount it)
data_path = 'ssh_login_features.csv'
df[feature_cols + ['entity_id']].to_csv(data_path, index=False)
print(f'Saved {len(df)} rows to {data_path}')

# create a dataset record on the platform
dataset = openuba.create_dataset(
    name='ssh-login-features-30d',
    description='30 days of SSH login behavior features for 200 users',
    source_type='upload',
    format='csv',
)
dataset_id = dataset.get('id')
print(f'\nDataset created on platform:')
print(f'  dataset_id: {dataset_id}')
print(f'  name:       {dataset.get("name")}')

---
## 8. Launch a Training Job

Submit a training job to the platform. The model runner will:
1. Load `MODEL.py` from the model library directory
2. Instantiate the `Model` class (which loads `model.pkl`)
3. Call `model.train(ctx)` with the dataset
4. Save the trained artifact

The generated `MODEL.py` follows the v2 interface that all models implement:
```python
class Model:
    def __init__(self):    # loads model.pkl
    def train(self, ctx):  # ctx.df = DataFrame, ctx.logger = logger
    def infer(self, ctx):  # returns DataFrame with risk scores
```

In [ ]:
training_job = openuba.start_training(
    model_id=model_id,
    dataset_id=dataset_id,
    hardware_tier='cpu-small',
    data_source='elasticsearch',
    index_name='openuba-proxy-toy_1',
    hyperparameters={
        'contamination': 0.05,
        'n_estimators': 200,
    },
)

training_job_id = training_job.get('id')
print(f'Training job submitted:')
print(f'  job_id:   {training_job_id}')
print(f'  status:   {training_job.get("status")}')
print(f'  hardware: {training_job.get("hardware_tier", "cpu-small")}')

In [ ]:
# wait for the training job to complete
if training_job_id:
    print('Waiting for training job to complete...')
    result = openuba.wait_for_job(training_job_id, poll_interval=3, timeout=300)
    print(f'Training finished with status: {result.get("status")}')

    # fetch logs
    try:
        logs = openuba.get_logs(training_job_id)
        print(f'\nTraining logs ({len(logs)} entries):')
        for entry in logs[-5:]:
            print(f'  [{entry.get("level", "info")}] {entry.get("message", "")}')
    except Exception as e:
        print(f'Could not fetch logs: {e}')
else:
    print('No training job ID — check previous cell for errors.')

---
## 9. Launch an Inference Job

Run inference on the same dataset. The model runner calls `model.infer(ctx)`
which returns a DataFrame with `entity_id`, `risk_score`, `anomaly_type`,
and `details` for each row.

In [ ]:
inference_job = openuba.start_inference(
    model_id=model_id,
    dataset_id=dataset_id,
    hardware_tier='cpu-small',
    data_source='elasticsearch',
    index_name='openuba-proxy-toy_1',
)

inference_job_id = inference_job.get('id')
print(f'Inference job submitted:')
print(f'  job_id: {inference_job_id}')
print(f'  status: {inference_job.get("status")}')

In [ ]:
if inference_job_id:
    print('Waiting for inference job to complete...')
    result = openuba.wait_for_job(inference_job_id, poll_interval=3, timeout=300)
    print(f'Inference finished with status: {result.get("status")}')

    try:
        logs = openuba.get_logs(inference_job_id)
        print(f'\nInference logs ({len(logs)} entries):')
        for entry in logs[-5:]:
            print(f'  [{entry.get("level", "info")}] {entry.get("message", "")}')
    except Exception as e:
        print(f'Could not fetch logs: {e}')
else:
    print('No inference job ID — check previous cell for errors.')

---
## 10. Publish a Version

After verifying the model works, publish a named version
so it can be referenced reproducibly.

In [ ]:
if model_id:
    version = openuba.publish_version(
        model_id=model_id,
        version='1.1.0',
        summary='Trained on 30d SSH login features (200 users, 5% anomaly rate). '
                f'Local eval: P={precision:.2f} R={recall:.2f} F1={f1:.2f}',
    )
    print(f'Version published:')
    for key, value in version.items():
        print(f'  {key}: {value}')
else:
    print('Skipped: no model_id from registration.')

---
## 11. Visualize Risk Scores

Create a risk score distribution visualization and publish it to the platform.
This will appear in **Visualizations** in the UI.

In [ ]:
import matplotlib.pyplot as plt

# compute risk scores (same logic as the generated MODEL.py wrapper)
risk_scores = []
for pred, score in zip(predictions, scores):
    if pred == -1:
        risk = min(100.0, abs(score) * 100 + 50)
    else:
        risk = max(0.0, (1 - score) * 20)
    risk_scores.append(risk)

df['risk_score'] = risk_scores

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# risk score distribution
normal_risk = df[df.label == 'normal']['risk_score']
anomalous_risk = df[df.label == 'anomalous']['risk_score']
ax1.hist(normal_risk, bins=30, alpha=0.6, label='Normal', color='steelblue')
ax1.hist(anomalous_risk, bins=30, alpha=0.6, label='Anomalous', color='crimson')
ax1.axvline(x=50, color='orange', linestyle='--', label='Threshold (50)')
ax1.set_xlabel('Risk Score')
ax1.set_ylabel('Count')
ax1.set_title('Risk Score Distribution')
ax1.legend()

# top 20 riskiest entities
top20 = df.nlargest(20, 'risk_score')
colors = ['crimson' if l == 'anomalous' else 'steelblue' for l in top20.label]
ax2.barh(top20.entity_id, top20.risk_score, color=colors)
ax2.set_xlabel('Risk Score')
ax2.set_title('Top 20 Riskiest Entities')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# publish the risk distribution chart to the platform
fig_pub, ax_pub = plt.subplots(figsize=(10, 5))
ax_pub.hist(normal_risk, bins=30, alpha=0.6, label='Normal', color='steelblue')
ax_pub.hist(anomalous_risk, bins=30, alpha=0.6, label='Anomalous', color='crimson')
ax_pub.axvline(x=50, color='orange', linestyle='--', label='Threshold')
ax_pub.set_xlabel('Risk Score')
ax_pub.set_ylabel('Count')
ax_pub.set_title('SSH Login Anomaly Risk Scores')
ax_pub.legend()
plt.tight_layout()

viz = openuba.create_visualization(
    name='ssh-login-risk-distribution',
    backend='matplotlib',
    description='Risk score distribution for SSH login anomaly detector',
    figure=fig_pub,
)
plt.close(fig_pub)

viz_id = viz.get('id')
print(f'Visualization created:')
print(f'  viz_id: {viz_id}')
print(f'  name:   {viz.get("name")}')
print(f'  status: {viz.get("status")}')

---
## 12. Create a Dashboard

Pin the visualization to a dashboard for ongoing monitoring.

In [ ]:
if viz_id:
    dashboard = openuba.create_dashboard(
        name='SSH Anomaly Detection',
        description='Monitors SSH login anomaly detection model outputs',
        layout=[
            {
                'i': str(viz_id),
                'x': 0, 'y': 0,
                'w': 12, 'h': 4,
                'visualization_id': str(viz_id),
            },
        ],
    )
    print(f'Dashboard created:')
    print(f'  dashboard_id: {dashboard.get("id")}')
    print(f'  name:         {dashboard.get("name")}')
    print(f'\nView it at: Dashboards > SSH Anomaly Detection')
else:
    print('Skipped: no visualization ID.')

---
## Summary

Here's what we accomplished:

In [ ]:
print('=== END-TO-END PIPELINE COMPLETE ===')
print()
print(f'1. Generated synthetic dataset:  {len(df)} users, {n_anomalous} anomalous')
print(f'2. Trained IsolationForest:       {clf.n_estimators} trees, contamination={clf.contamination}')
print(f'3. Local evaluation:              P={precision:.2f} R={recall:.2f} F1={f1:.2f}')
print(f'4. Registered model:              {handle.get("name")} -> {model_id}')
print(f'5. Model status:                  {handle.get("status")}')
print(f'6. Dataset created:               {dataset_id}')
print(f'7. Training job:                  {training_job_id}')
print(f'8. Inference job:                 {inference_job_id}')
if viz_id:
    print(f'9. Visualization published:       {viz_id}')
print()
print('The model is now visible in the UI under Models > Installed.')
print('You can retrain, run inference, or schedule it in a pipeline.')

---
## Next Steps

- **Experiment tracking:** Compare different contamination rates with `openuba.create_experiment()` (see `07_ml_workflow.ipynb`)
- **Hyperparameter tuning:** Save configs with `openuba.create_hyperparameters()` and sweep across them
- **Pipelines:** Chain preprocessing + training + inference with `openuba.create_pipeline()`
- **Custom MODEL.py:** Pass your own source code via `openuba.register_model(name, source_code=code)` for full control
- **PyTorch models:** The SDK auto-detects PyTorch and uses `torch.save()`/`torch.load()` — same workflow